<a href="https://colab.research.google.com/github/dheena-agri/DRL-Trading-Project/blob/main/DRL_Trading_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install yfinance gymnasium numpy pandas matplotlib torch

In [4]:
import yfinance as yf
import pandas as pd


# Download data
data = yf.download("SPY", start="2015-01-01", end="2024-01-01")


# Use closing price
data = data[['Close']]


# Calculate daily returns
data['Return'] = data['Close'].pct_change()
data.dropna(inplace=True)


data.head()

/tmp/ipython-input-2842501636.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download("SPY", start="2015-01-01", end="2024-01-01")
[*********************100%***********************]  1 of 1 completed


Price,Close,Return
Ticker,SPY,
Date,,
2015-01-05,167.508835,-0.018059
2015-01-06,165.931091,-0.009419
2015-01-07,167.998795,0.012461
2015-01-08,170.979874,0.017745
2015-01-09,169.609756,-0.008013


In [6]:
import gymnasium as gym
import numpy as np


class TradingEnv(gym.Env):
    def __init__(self, data, window=10):
        self.data = data
        self.window = window
        self.current_step = window
        self.position = 0 # 0 = no stock, 1 = holding
        self.cash = 1.0
        self.shares = 0.0


        self.action_space = gym.spaces.Discrete(3)
        self.observation_space = gym.spaces.Box(
            low=-1, high=1, shape=(window,), dtype=np.float32
        )


    def reset(self):
        self.current_step = self.window
        self.position = 0
        self.cash = 1.0
        self.shares = 0.0
        return self._get_state(), {}


    def _get_state(self):
        return self.data['Return'].values[
            self.current_step-self.window:self.current_step
        ]


    def step(self, action):
        price = self.data['Close'].iloc[self.current_step]
        reward = 0


        if action == 1 and self.position == 0: # Buy
            self.shares = self.cash / price
            self.cash = 0
            self.position = 1


        elif action == 2 and self.position == 1: # Sell
            self.cash = self.shares * price
            self.shares = 0
            self.position = 0


        portfolio_value = self.cash + self.shares * price
        reward = portfolio_value - 1.0


        self.current_step += 1
        done = self.current_step >= len(self.data) - 1


        return self._get_state(), reward, done, False, {}

In [8]:
import torch
import torch.nn as nn


class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )


    def forward(self, x):
        return self.net(x)

In [10]:
import random
from collections import deque


class ReplayBuffer:
    def __init__(self, capacity=10000):
      self.buffer = deque(maxlen=capacity)


    def push(self, state, action, reward, next_state, done):
      self.buffer.append((state, action, reward, next_state, done))


    def sample(self, batch_size):
      batch = random.sample(self.buffer, batch_size)
      return map(np.array, zip(*batch))


    def __len__(self):
      return len(self.buffer)

In [11]:
LR = 1e-3
GAMMA = 0.99
BATCH_SIZE = 32
EPISODES = 200

In [17]:
env = TradingEnv(data)
model = DQN(10, 3)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
buffer = ReplayBuffer()


rewards_history = []


for episode in range(EPISODES):
    state, _ = env.reset()
    total_reward = 0


    while True:
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        action = torch.argmax(model(state_tensor)).item()


        next_state, reward, done, _, _ = env.step(action)
        buffer.push(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward


        if done:
            break


    rewards_history.append(total_reward.item())


    if episode % 20 == 0:
        print(f"Episode {episode}, Reward: {total_reward.item():.4f}")

Episode 0, Reward: 1589.5237
Episode 20, Reward: 1589.5237
Episode 40, Reward: 1589.5237
Episode 60, Reward: 1589.5237
Episode 80, Reward: 1589.5237
Episode 100, Reward: 1589.5237
Episode 120, Reward: 1589.5237
Episode 140, Reward: 1589.5237
Episode 160, Reward: 1589.5237
Episode 180, Reward: 1589.5237


In [15]:
buy_hold_return = data['Close'].iloc[-1] / data['Close'].iloc[0] - 1
print("Buy & Hold Return:", buy_hold_return)

Buy & Hold Return: Ticker
SPY    1.769068
dtype: float64


In [18]:
returns = pd.Series(rewards_history)
sharpe = returns.mean() / returns.std()
print("Sharpe Ratio:", sharpe)

Sharpe Ratio: 498092870448644.0


In [19]:
cum_returns = returns.cumsum()
drawdown = cum_returns - cum_returns.cummax()
print("Max Drawdown:", drawdown.min())

Max Drawdown: 0.0
